# Introduction to Human-Computer Interaction and how Artificial Intelligence and Computer Vision play a role in it

Welcome to the evolution of Human-Computer Interaction (HCI). Traditionally, HCI relied on mechanical peripherals: keyboards, mice, and touchscreens. These require the human to adapt to the machine's input language. 

With the integration of **Artificial Intelligence (AI)** and **Computer Vision (CV)**, the paradigm is shifting. Machines are now learning to understand natural human behaviors—such as voice, eye movement, and hand gestures—making interactions frictionless and intuitive. 

In this analytical tutorial, we will build a core component of modern HCI: **A Non-Touch Interface via Hand Tracking**. We will use `OpenCV` for image processing and Google's `MediaPipe`, an applied ML pipeline, to extract structural data from a human hand and translate it into a computer command (a "click").

In [ ]:
# Cell 2: Environment Setup
# Install required libraries if you haven't already:
# !pip install opencv-python mediapipe numpy matplotlib

import cv2
import mediapipe as mp
import numpy as np
import math

# We use matplotlib to display images inline if running in a Jupyter Notebook
import matplotlib.pyplot as plt

print("Libraries imported successfully. Ready for HCI pipeline.")

## Step 1: The ML Pipeline Architecture (MediaPipe)

Extracting hand gestures from a 2D image is analytically complex. MediaPipe simplifies this by using a two-stage machine learning pipeline:
1. **Palm Detection Model:** Scans the full image and returns an oriented hand bounding box.
2. **Hand Landmark Model:** Operates on the cropped bounding box and predicts 21 3D landmarks (knuckles, fingertips, etc.).

This two-stage approach saves computational power, allowing real-time HCI on standard CPUs.

In [ ]:
# Cell 4: Initializing the AI Models

# Initialize the MediaPipe Hands module
mp_hands = mp.solutions.hands

# Configure the model parameters:
# static_image_mode=True (for processing single images rather than a video stream)
# max_num_hands=1 (we only need one hand for a mouse pointer)
# min_detection_confidence=0.5 (threshold for the AI to consider a detection valid)
hands_model = mp_hands.Hands(
    static_image_mode=True, 
    max_num_hands=1, 
    min_detection_confidence=0.5
)

# MediaPipe also provides a drawing utility to visualize the landmarks easily
mp_drawing = mp.solutions.drawing_utils

print("MediaPipe Hand Tracking AI initialized.")

## Step 2: Translating Pixels to Spatial Coordinates

In standard HCI (like a mouse), the hardware sends X and Y coordinates to the operating system. In our CV approach, the camera provides an array of RGB pixels. The AI model's job is to map those pixels to a coordinate system.

Let's simulate processing a frame. (Note: For this notebook tutorial, we will create a dummy canvas, but the logic is exactly the same for a webcam frame captured via `cv2.VideoCapture`).

In [ ]:
# Cell 6: Processing an Image Frame

def process_hand_frame(image_array):
    """
    Takes an OpenCV image (BGR format), converts it to RGB, 
    and passes it through the ML model to extract coordinates.
    """
    # OpenCV loads images in BGR, but MediaPipe requires RGB.
    image_rgb = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
    
    # Process the image through the neural network
    results = hands_model.process(image_rgb)
    
    # Create a copy of the image to draw annotations on
    annotated_image = image_array.copy()
    
    landmarks_data = []
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Draw the skeletal structure on the image
            mp_drawing.draw_landmarks(
                annotated_image, 
                hand_landmarks, 
                mp_hands.HAND_CONNECTIONS
            )
            
            # Extract the actual coordinates
            # Coordinates are normalized [0.0, 1.0], so we multiply by image dimensions
            h, w, c = image_array.shape
            for id, lm in enumerate(hand_landmarks.landmark):
                cx, cy = int(lm.x * w), int(lm.y * h)
                landmarks_data.append((id, cx, cy))
                
    return annotated_image, landmarks_data

print("Frame processing pipeline defined.")

## Step 3: Analytical HCI Logic (Detecting a "Pinch" Click)

Now we have the 21 spatial coordinates of the hand. How do we turn this into a computer command? 
We define a heuristic. A common gesture for "clicking" or "grabbing" in Spatial Computing (like Apple Vision Pro) is bringing the thumb tip and index finger tip together.

In MediaPipe, the Thumb Tip is landmark ID `4`, and the Index Finger Tip is ID `8`. 

We calculate the Euclidean distance $d$ between these two points:
$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

If $d$ drops below a certain threshold, we register a "Click".

In [ ]:
# Cell 8: Implementing the HCI Logic

def detect_click(landmarks_data, threshold=40):
    """
    Analyzes the structural data of the hand to detect a pinch gesture.
    """
    if not landmarks_data:
        return False, 0
    
    # Extract coordinates for Thumb Tip (4) and Index Finger Tip (8)
    # Using list comprehension to find the specific tuples
    thumb = next((item for item in landmarks_data if item[0] == 4), None)
    index = next((item for item in landmarks_data if item[0] == 8), None)
    
    if thumb and index:
        x1, y1 = thumb[1], thumb[2]
        x2, y2 = index[1], index[2]
        
        # Calculate Euclidean Distance
        distance = math.hypot(x2 - x1, y2 - y1)
        
        # Determine if it's a click based on the analytical threshold
        is_click = distance < threshold
        return is_click, distance
        
    return False, 0

print("HCI translation logic defined.")

## Conclusion: The Bridge between AI and User Experience

By executing the pipeline above, we have successfully bridged the gap between human anatomy and machine instructions. 

1. **Computer Vision** captured the environment.
2. **Artificial Intelligence (MediaPipe)** processed the noisy pixel data and found structured mathematical patterns (landmarks).
3. **Analytical Logic** interpreted those patterns using geometry (Euclidean distance) to fire an HCI event (a click).

To take this into a real-time production environment, you would place this logic inside a `cv2.VideoCapture(0)` while loop, allowing you to control your computer's actual mouse using a library like `pyautogui` based entirely on thin air!